# Kathmandu traffic vision - train plate reader

Trains the two models of the **plate reader**. (1) A character detector (YOLO11s, one class) on 2,503 cropped Nepali plates with character boxes. (2) A character classifier (YOLO11s-cls) on single-character crops: Devanagari letters and digits, and embossed-plate digits and Latin letters. Each stage is scored on held-out data, per source. No public dataset gives whole-plate text, so end-to-end reading accuracy cannot be measured here; that needs hand-labelled plates from Kathmandu footage.

In [ ]:
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:

import os, random, yaml
from pathlib import Path
IN, OUT = Path("/kaggle/input/datasets"), Path("/kaggle/working/data")
VAL_LISTS = {}

def label_of(img):
    parts = list(img.parts)
    if "images" in parts:                                       # YOLO layout: .../images/... -> .../labels/...
        i = len(parts) - 1 - parts[::-1].index("images")
        p = Path(*parts[:i], "labels", *parts[i + 1:]).with_suffix(".txt")
        if p.exists():
            return p
    return img.with_suffix(".txt")                              # some sets keep labels next to images

def add(tag, split, images, names, mapping, repeat=1):
    """Link images into OUT/<split>, rewriting labels: keep only source classes in `mapping`
    (source name -> target id); boxes of other classes are dropped, the image stays."""
    d = OUT / split
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)
    n_box, kept = 0, []
    for img in images:
        rows, lab = [], label_of(img)
        if lab.exists():
            for ln in lab.read_text(errors="ignore").splitlines():
                parts = ln.split()
                if len(parts) != 5 or not parts[0].isdigit():      # skip polygons / malformed rows
                    continue
                k = int(parts[0])
                if k < len(names) and names[k] in mapping:
                    rows.append(" ".join([str(mapping[names[k]])] + parts[1:]))
        for r in range(repeat):
            stem = f"{tag}_{r}_{img.parent.name}_{img.stem}".replace(" ", "_")
            link = d / "images" / (stem + img.suffix.lower())
            if not link.exists():
                link.symlink_to(img)
            (d / "labels" / (stem + ".txt")).write_text("\n".join(rows))
            kept.append(str(link))
        n_box += len(rows)
    if split == "val":
        VAL_LISTS[tag] = kept
    print(f"{tag:<10} {split:<5} {len(images):>6} images x{repeat}  {n_box:>7} boxes kept")

def imgs(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

def split_random(images, val_share=0.15, seed=0):
    images = images[:]
    random.Random(seed).shuffle(images)
    n = int(len(images) * val_share)
    return images[n:], images[:n]

def write_yaml(names):
    (OUT / "data.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "val/images",
                                                 "names": dict(enumerate(names))}))
    for tag, files in VAL_LISTS.items():          # per-source validation sets, to see where the model is weak
        (OUT / f"val_{tag}.txt").write_text("\n".join(files))
        (OUT / f"data_{tag}.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images",
                                                            "val": f"val_{tag}.txt", "names": dict(enumerate(names))}))


In [ ]:

cha = IN / "inspiring-lab/nepali-vehicles-number-plate-dataset/open_source_nepali_plates"
tr, va = split_random(imgs(cha / "images"))
add("plates_dev", "train", tr, ["character"], {"character": 0})
add("plates_dev", "val", va, ["character"], {"character": 0})
write_yaml(["character"])

from ultralytics import YOLO
import glob, shutil
det = YOLO("yolo11s.pt")
det.train(data=str(OUT / "data.yaml"), epochs=80, time=1.5, imgsz=416, batch=32, patience=15,
          project="/kaggle/working/runs", name="chars", workers=4, seed=0, plots=True)
shutil.copy(sorted(glob.glob("/kaggle/working/runs/chars*/weights/best.pt"))[-1], "/kaggle/working/chars_best.pt")
r = YOLO("/kaggle/working/chars_best.pt").val(data=str(OUT / "data.yaml"), imgsz=416, verbose=False, plots=False)
print(f"character detector: mAP50 {r.box.map50:.3f}  mAP50-95 {r.box.map:.3f}  "
      f"precision {r.box.mp:.3f}  recall {r.box.mr:.3f}  ({len(va)} validation plates)")
shutil.rmtree(OUT)


In [ ]:

import random, shutil, glob
from collections import Counter
from pathlib import Path
from ultralytics import YOLO
CLS = Path("/kaggle/working/glyphs")
VAL = {}     # source -> [(image link, class)]

def link_all(tag, pairs, split):
    for i, (img, c) in enumerate(pairs):
        d = CLS / split / c
        d.mkdir(parents=True, exist_ok=True)
        dst = d / f"{tag}_{i}{img.suffix.lower()}"
        if not dst.exists():
            dst.symlink_to(img)
        if split == "val":
            VAL.setdefault(tag, []).append((dst, c))
    print(f"{tag:<10} {split:<5} {len(pairs):>6} crops, {len(set(c for _, c in pairs))} classes")

# Devanagari: one folder per character, no split given -> random 15% validation
dev = IN / "inspiring-lab/nepali-number-plate-characters-dataset/character_ocr"
pairs = [(p, p.parent.name) for p in imgs(dev)]
random.Random(0).shuffle(pairs)
n = int(len(pairs) * 0.15)
link_all("devanagari", pairs[n:], "train")
link_all("devanagari", pairs[:n], "val")

# Embossed: class folders under train/ (2,670), valid/ (349) and test/ (563); the flat augmented/ folder
# (6,566 files, no class in path or name) is skipped. utsavbaral007's set has the same test folders: a copy, not used.
emb = IN / "theciceerguy/embossed-number-plate-detection-nepal-dataset/digits_and_numbers_dataset"
name = lambda p: "flag" if p.parent.name == "Nepali Flag" else "emb_" + p.parent.name
link_all("embossed", [(p, name(p)) for p in imgs(emb / "train")], "train")
link_all("embossed", [(p, name(p)) for p in imgs(emb / "valid") + imgs(emb / "test")], "val")


cls = YOLO("yolo11s-cls.pt")
cls.train(data=str(CLS), epochs=40, time=1.5, imgsz=64, batch=256, patience=10,
          project="/kaggle/working/runs", name="glyphs", workers=4, seed=0, plots=True)
shutil.copy(sorted(glob.glob("/kaggle/working/runs/glyphs*/weights/best.pt"))[-1], "/kaggle/working/glyphs_best.pt")

m = YOLO("/kaggle/working/glyphs_best.pt")
rows = []
for tag, items in VAL.items():
    wrong = Counter()
    ok = 0
    for i in range(0, len(items), 512):
        batch = items[i:i + 512]
        for (img, c), r in zip(batch, m.predict([str(p) for p, _ in batch], imgsz=64, verbose=False)):
            got = r.names[r.probs.top1]
            ok += got == c
            if got != c:
                wrong[f"{c} -> {got}"] += 1
    rows.append({"source": tag, "crops": len(items), "top1": round(ok / len(items), 4),
                 "most_confused": "; ".join(f"{k} ({v})" for k, v in wrong.most_common(5))})
import pandas as pd
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/glyphs_per_source.csv", index=False)
print(res.to_string(index=False))
shutil.rmtree(CLS)
